# Advanced 11 — Compliance, Audit & Forensic Readiness for Agent Identity

**Enterprise scenario:** you are preparing an autonomous claims-agent platform for continuous internal assurance and an external audit. The system includes workload identities, delegated sub-agents, tool permissions, external integrations and high-risk production actions.

Your job is to prove controls—not merely describe them.


In [ ]:
import pandas as pd, json, hashlib, random, networkx as nx
from datetime import datetime,timedelta,timezone
NOW=datetime.now(timezone.utc)


## Lab 1 — Build a control catalog

In [ ]:
controls=pd.DataFrame([
{"id":"AG-01","name":"Registered identity","domain":"governance","critical":True},
{"id":"AG-02","name":"Accountable owner","domain":"governance","critical":True},
{"id":"AC-01","name":"Least privilege","domain":"authorization","critical":False},
{"id":"CR-01","name":"Short-lived production credential","domain":"credential","critical":True},
{"id":"DG-01","name":"Delegation attenuation","domain":"delegation","critical":True},
{"id":"AU-01","name":"Critical action evidence","domain":"audit","critical":True}])
controls

## Lab 2 — Map controls to frameworks

In [ ]:
mapping={"AG-01":["NIST IA","CIS 5"],"AC-01":["NIST AC","CIS 6","OWASP NHI5"],
"CR-01":["NIST IA","OWASP NHI7"],"AU-01":["NIST AU","CIS 8"]}
mapping

## Lab 3 — Load identity inventory

In [ ]:
inventory=pd.DataFrame([
{"id":"agent:claims","owner":"claims-ai","env":"prod","risk":"high","credential":"workload","lifetime":10},
{"id":"agent:research","owner":"claims-ai","env":"prod","risk":"high","credential":"workload","lifetime":15},
{"id":"agent:legacy","owner":"","env":"prod","risk":"critical","credential":"static_api_key","lifetime":525600}])
inventory

## Lab 4 — Determine control applicability

In [ ]:
applicability=pd.DataFrame([
{"agent":"agent:claims","control":"CR-01","applicable":True,"rationale":"production"},
{"agent":"agent:claims","control":"DG-01","applicable":True,"rationale":"delegates"},
{"agent":"agent:research","control":"DG-01","applicable":False,"rationale":"redelegation disabled"}])
applicability

## Lab 5 — Test accountable ownership

In [ ]:
inventory.assign(pass_owner=inventory.owner.str.len()>0)

## Lab 6 — Risk-tier requirements

In [ ]:
requirements={"low":{"review_days":365},"medium":{"review_days":180},"high":{"review_days":90},"critical":{"review_days":30}}
requirements

## Lab 7 — Credential compliance

In [ ]:
inventory.assign(credential_pass=~((inventory.env=="prod")&((inventory.credential=="static_api_key")|(inventory.lifetime>60))))

## Lab 8 — Least-privilege comparison

In [ ]:
granted={"agent:research":{"documents.read","claims.read"}}
used={"agent:research":{"documents.read"}}
{"unused":granted["agent:research"]-used["agent:research"]}

## Lab 9 — Delegation attenuation

In [ ]:
parent={"documents.read"};child={"documents.read","claims.write"}
{"pass":child.issubset(parent),"excess":child-parent}

## Lab 10 — Segregation-of-duties graph

In [ ]:
roles={"alice":{"create_agent"},"bob":{"approve_agent"},"carol":{"write_policy","approve_policy"}}
toxic=[{"write_policy","approve_policy"},{"create_agent","approve_agent"}]
[(u,t) for u,p in roles.items() for t in toxic if t<=p]

## Lab 11 — Offboarding completeness

In [ ]:
offboard={"identity_disabled":True,"credentials_revoked":True,"delegations_removed":True,
"tool_grants_removed":True,"runtime_stopped":False}
{"pass":all(offboard.values()),"missing":[k for k,v in offboard.items() if not v]}

## Lab 12 — Third-party assurance

In [ ]:
third_party={"provider":"partner-ai","owner":"vendor-risk","identity":"federated",
"permissions_reviewed":True,"incident_contact":True,"offboarding_tested":False}
third_party

## Lab 13 — Policy-as-code invariant

In [ ]:
def compliant(a):
    if a["env"]=="prod" and (not a["owner"] or a["credential"]=="static_api_key" or a["lifetime"]>60):return False
    return True
[(r.id,compliant(r._asdict())) for r in inventory.itertuples(index=False)]

## Lab 14 — CI/CD compliance gate

In [ ]:
results={"registration":True,"owner":True,"credential":True,"policy_tests":True,"observability":False}
{"deploy":all(results.values()),"failed":[k for k,v in results.items() if not v]}

## Lab 15 — Evidence specification

In [ ]:
evidence_spec={"CR-01":["credential_config","issuance_events","runtime_validation"],
"AU-01":["pdp_decision","pep_enforcement","resource_action"]}
evidence_spec

## Lab 16 — Collect evidence artifact

In [ ]:
def artifact(name,source,data):
    raw=json.dumps(data,sort_keys=True).encode()
    return {"name":name,"source":source,"collected":NOW.isoformat(),"sha256":hashlib.sha256(raw).hexdigest()}
artifact("agent_inventory","identity-registry",inventory.to_dict("records"))

## Lab 17 — Build evidence manifest

In [ ]:
artifacts=[artifact("inventory","registry",inventory.to_dict("records")),
artifact("controls","grc",controls.to_dict("records"))]
manifest={"period":"2026-Q3","artifacts":artifacts}
manifest

## Lab 18 — Evidence freshness

In [ ]:
evidence=pd.DataFrame([
{"name":"inventory","age_hours":3,"max_hours":24},{"name":"owner_attestation","age_hours":1200,"max_hours":2160},
{"name":"credential_status","age_hours":12,"max_hours":1}])
evidence.assign(fresh=evidence.age_hours<=evidence.max_hours)

## Lab 19 — Evidence completeness

In [ ]:
population=100;tested=94;exceptions=2;missing=population-tested
{"population":population,"tested":tested,"coverage":tested/population,"exceptions":exceptions,"missing":missing}

## Lab 20 — Continuous control monitoring

In [ ]:
def monitor(df):
    return pd.DataFrame([{"agent":r.id,"owner_ok":bool(r.owner),"credential_ok":not(r.env=="prod" and (r.credential=="static_api_key" or r.lifetime>60))}
                         for r in df.itertuples()])
monitor(inventory)

## Lab 21 — Owner attestation workflow

In [ ]:
attestation={"agent":"agent:research","owner":"claims-ai","purpose_valid":True,
"permissions_justified":False,"delegations_justified":True,"attested_at":NOW.isoformat()}
attestation

## Lab 22 — Access certification

In [ ]:
cert=pd.DataFrame([
{"identity":"agent:claims","grant":"claims.read","decision":"retain"},
{"identity":"agent:research","grant":"claims.read","decision":"remove"}])
cert

## Lab 23 — Event-driven recertification

In [ ]:
events=["owner_changed","new_privileged_tool","documentation_update"]
review_triggers={"owner_changed","new_privileged_tool","scope_expanded","incident","new_federation"}
[x for x in events if x in review_triggers]

## Lab 24 — Exception workflow

In [ ]:
exception={"id":"EX-12","control":"CR-01","subject":"agent:legacy","risk":"high",
"compensating":["vault","network restriction","enhanced monitoring"],"expires":NOW+timedelta(days=14),"approved":True}
exception

## Lab 25 — Validate compensating controls

In [ ]:
required={"vault","network restriction","enhanced monitoring"}
required.issubset(set(exception["compensating"]))

## Lab 26 — Detect expired exceptions

In [ ]:
exceptions=[{"id":"e1","expires":NOW-timedelta(days=1)},{"id":"e2","expires":NOW+timedelta(days=10)}]
[x["id"] for x in exceptions if x["expires"]<NOW]

## Lab 27 — Configuration drift

In [ ]:
approved={"scope":{"documents.read"},"credential":"workload"}
deployed={"scope":{"documents.read","claims.write"},"credential":"workload"}
{"scope_drift":deployed["scope"]-approved["scope"],"credential_drift":deployed["credential"]!=approved["credential"]}

## Lab 28 — Define audit universe

In [ ]:
universe=pd.DataFrame([{"id":f"agent:{i}","risk":random.choice(["low","medium","high","critical"])} for i in range(1,101)])
universe.risk.value_counts()

## Lab 29 — Random sampling

In [ ]:
universe.sample(10,random_state=42)

## Lab 30 — Risk-based sampling

In [ ]:
high=universe[universe.risk.isin(["high","critical"])]
sample=pd.concat([high.sample(min(15,len(high)),random_state=1),universe[universe.risk=="low"].sample(3,random_state=2)])
sample

## Lab 31 — Control test record

In [ ]:
test_record={"control":"CR-01","population":100,"sample":18,"procedure":"inspect credential type/lifetime",
"result":"exception","tester":"internal-audit","tested_at":NOW.isoformat(),"evidence":["manifest:2026Q3"]}
test_record

## Lab 32 — Create finding

In [ ]:
finding={"id":"F-17","control":"CR-01","severity":"high","subject":"agent:legacy",
"root_cause":"legacy integration","owner":"claims-platform","due":(NOW+timedelta(days=30)).date().isoformat(),"status":"open"}
finding

## Lab 33 — Remediation and retest

In [ ]:
finding["status"]="implemented"
retest={"finding":"F-17","test":"credential lifetime <=60","result":"pass","tested_at":NOW.isoformat()}
finding,retest

## Lab 34 — Risk-weighted posture

In [ ]:
control_results=pd.DataFrame([
{"control":"AG-01","pass":1,"weight":5},{"control":"AG-02","pass":0,"weight":5},
{"control":"AC-01","pass":1,"weight":3},{"control":"CR-01","pass":0,"weight":5},{"control":"AU-01","pass":1,"weight":5}])
100*(control_results["pass"]*control_results.weight).sum()/control_results.weight.sum()

## Lab 35 — Critical override

In [ ]:
critical_failures=["missing_owner","static_prod_credential"]
aggregate=78
effective=min(aggregate,25) if critical_failures else aggregate
effective

## Lab 36 — Executive dashboard

In [ ]:
dashboard={"critical_agents_controlled_pct":82,"critical_failures":2,"expired_exceptions":1,
"overdue_reviews":4,"evidence_coverage_pct":94,"high_findings":3}
dashboard

## Lab 37 — Retention policy

In [ ]:
retention={"routine_trace_days":30,"security_event_days":365,"audit_evidence_days":2555,
"sensitive_content_days":7}
retention

## Lab 38 — Legal hold

In [ ]:
hold={"id":"LH-44","case":"IR-2026-17","scope":["trace-42","agent:research","dlg-9"],
"start":NOW.isoformat(),"status":"active","overrides_deletion":True}
hold

## Lab 39 — Chain of custody

In [ ]:
custody=pd.DataFrame([
{"step":1,"actor":"collector","action":"collect","hash":"abc"},
{"step":2,"actor":"evidence-service","action":"archive","hash":"abc"},
{"step":3,"actor":"auditor","action":"export-copy","hash":"abc"}])
custody

## Lab 40 — Historical authority reconstruction

In [ ]:
history=[
{"date":"2026-07-01","agent":"research","scope":{"documents.read"},"policy":"16"},
{"date":"2026-08-01","agent":"research","scope":{"documents.read","claims.read"},"policy":"17"}]
[x for x in history if x["date"]<="2026-08-15"][-1]

## Lab 41 — Generate audit pack

In [ ]:
audit_pack={"scope":"production agent identities","period":"2026-Q3",
"controls":controls.to_dict("records"),"evidence_manifest":manifest,
"exceptions":[{"id":"EX-12"}],"findings":[finding],"generated_at":NOW.isoformat()}
len(json.dumps(audit_pack,default=str))

## Lab 42 — Third-party evidence review

In [ ]:
vendor_evidence={"identity_architecture":True,"incident_process":True,"offboarding_test":False,
"credential_lifecycle":True,"audit_capability":True}
[k for k,v in vendor_evidence.items() if not v]

## Lab 43 — Inherited controls

In [ ]:
inheritance=pd.DataFrame([
{"control":"workload identity","provider":"agent-platform","consumer":"claims-agent","inherited":True},
{"control":"business authorization","provider":"claims-team","consumer":"claims-agent","inherited":False}])
inheritance

## Lab 44 — Assurance trend

In [ ]:
trend=pd.DataFrame({"month":["Apr","May","Jun","Jul","Aug"],"score":[68,72,75,81,86],
"critical_findings":[6,5,4,2,1]})
trend

# Lab 44 — Capstone: Continuous Assurance for an Enterprise Agent Identity Platform

Build an assurance pipeline for:

```text
Human users
    ↓
Claims Orchestrator
    ├── Research Agent
    └── Data Agent
          ↓
     MCP / Tools
          ↓
 Cloud / Claims APIs
          ↓
      Sensitive Data
```

The platform also has:

- workload identities;
- OAuth/token exchange;
- delegated authority;
- an external partner agent;
- KMS-backed operations;
- policy-as-code;
- identity telemetry;
- one legacy static credential;
- one expired exception;
- one orphaned agent;
- one overprivileged delegation;
- one retired agent that can still execute.

Your system must:

1. establish the complete audit universe;
2. assign control applicability;
3. evaluate all machine-testable controls;
4. map controls to relevant frameworks;
5. identify critical failures;
6. detect segregation-of-duties conflicts;
7. evaluate credential posture;
8. verify delegation attenuation;
9. verify offboarding;
10. evaluate third-party assurance;
11. generate evidence artifacts;
12. hash and manifest the evidence;
13. measure freshness and completeness;
14. process attestations;
15. detect expired exceptions;
16. verify compensating controls;
17. compare approved/deployed/runtime state;
18. create findings;
19. generate remediation and retest records;
20. build a risk-weighted assurance score;
21. apply critical-condition overrides;
22. generate an audit pack;
23. build an executive assurance summary;
24. demonstrate historical authority reconstruction;
25. demonstrate legal-hold and chain-of-custody handling.

**Success criterion:** an auditor should be able to move from a framework requirement to a control, implementation, test, population, result and verifiable evidence without relying on screenshots or tribal knowledge.


# Review questions

1. What is the difference between a requirement, objective, control, test and evidence?
2. How do preventive, detective and corrective controls differ?
3. When should a control be automated?
4. Why is inventory completeness foundational to audit?
5. How should control applicability be documented?
6. How do design and operating effectiveness differ?
7. Why should NHI access certification include delegations and tool grants?
8. How can segregation of duties be graph-tested?
9. What makes evidence trustworthy?
10. Why should missing evidence not be treated as a pass?
11. What is evidence freshness?
12. Why can screenshots be weak continuous-control evidence?
13. What is a compensating control?
14. What makes an exception governable?
15. When is 100% testing preferable to sampling?
16. Why should high-risk populations be oversampled?
17. Why can aggregate compliance scores be misleading?
18. What is forensic readiness?
19. Why must historical policy state be preserved?
20. How should inherited platform controls be documented?
